# 05 - Skill Memory: scientific Split CIFAR-100 evaluation

This notebook replaces the old synthetic Skill Memory demonstration as the research-facing analysis. It consumes the five-seed Split CIFAR-100 artifacts produced by `scientific-comparison.yml` and inspects final accuracy, causal forgetting, and the REUSE/CLONE/SCRATCH decisions.

Protocol: 20 experiences, memory size 2000, seeds 0--4. The primary evaluation is the active model; the plugin's labeled `before_eval_exp` retrieval hook is not used here.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

root=Path.cwd().resolve()
if not (root/'results').exists(): root=root.parent
results=root/'results'/'skill_memory_split_cifar100'
if not results.exists(): raise FileNotFoundError('Run .github/workflows/scientific-comparison.yml first.')
rows=[]; decisions=[]
for path in sorted(results.glob('*/summary.json')):
    d=json.loads(path.read_text())
    rows.append({k:d[k] for k in ['seed','final_accuracy','forgetting','AAA_test','evaluation_mode'] if k in d})
    for decision in d.get('decisions',[]): decisions.append({'seed':d['seed'],**decision})
replicates=pd.DataFrame(rows).sort_values('seed')
display(replicates)
print('Seeds:', sorted(replicates.seed.tolist()))
print('Final accuracy: %.2f%% +/- %.2f%%' % (100*replicates.final_accuracy.mean(),100*replicates.final_accuracy.std()))
print('Forgetting: %.2f%% +/- %.2f%%' % (100*replicates.forgetting.mean(),100*replicates.forgetting.std()))


## Acquisition policy diagnostics

The decision log is part of the result artifact rather than a manually selected example. This lets the research comparison distinguish actual REUSE/CLONE/SCRATCH behavior from the final accuracy numbers.


In [ ]:
decisions=pd.DataFrame(decisions)
if decisions.empty:
    raise RuntimeError('No decision logs found.')
display(pd.crosstab(decisions.seed, decisions.decision, margins=True))
display(decisions.groupby('decision')[['compatibility_score','old_accuracy','new_accuracy']].mean())


## Replicate distributions


In [ ]:
plt.figure(figsize=(8,5))
plt.errorbar(['Skill Memory'],[100*replicates.final_accuracy.mean()],[100*replicates.final_accuracy.std()],fmt='o')
plt.ylabel('Final test accuracy (%)')
plt.title('Skill Memory: five-seed Split CIFAR-100 result')
plt.tight_layout(); plt.show()

plt.figure(figsize=(8,5))
plt.errorbar(['Skill Memory'],[100*replicates.forgetting.mean()],[100*replicates.forgetting.std()],fmt='o')
plt.ylabel('Causal average forgetting (%)')
plt.title('Skill Memory: five-seed forgetting result')
plt.tight_layout(); plt.show()
